In [7]:
import sys

sys.path.append("../..")
import torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from interpreto.attributions import Saliency
from interpreto.concepts.metrics import AttrSim
from interpreto.model_wrapping.llm_interface import HuggingFaceLLM

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "textattack/distilbert-base-uncased-ag-news"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
dataset = load_dataset("fancyzhx/ag_news")

n_train = 700
train_inputs = dataset["train"]["text"][:n_train]
classes_names = dataset["train"].features["label"].names

In [9]:
def predict_in_batches(model, tokenizer, texts, device, batch_size=32):
    model.eval()
    predictions = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]

            encoded = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True)
            encoded = {k: v.to(device) for k, v in encoded.items()}

            outputs = model(**encoded)
            preds = outputs.logits.argmax(dim=-1).cpu()
            predictions.append(preds)

            del encoded, outputs, preds
            torch.cuda.empty_cache()

    return torch.cat(predictions, dim=0)


train_predictions = predict_in_batches(model, tokenizer, train_inputs, device, batch_size=32)

In [10]:
attrsim = AttrSim(classes=classes_names)
train_labels = torch.tensor(dataset["train"]["label"][:n_train])

# Select a balanced pool (good/miss) then force 10 good + 10 miss
# for the ConSim learning phase.
indices, samples, selected_labels, selected_predictions = attrsim.select_examples(
    inputs=train_inputs,
    labels=train_labels,
    predictions=train_predictions,
    nb_samples=30,
    seed=0,
)

good_mask = selected_labels == selected_predictions
good_idx = torch.where(good_mask)[0]
miss_idx = torch.where(~good_mask)[0]

lp_good = good_idx[:10]
lp_miss = miss_idx[:10]
lp_idx = torch.cat([lp_good, lp_miss])

all_idx = torch.arange(len(samples))
ep_idx = all_idx[~torch.isin(all_idx, lp_idx)]
ordered_idx = torch.cat([lp_idx, ep_idx])

samples = [samples[i] for i in ordered_idx.tolist()]
selected_labels = selected_labels[ordered_idx]
selected_predictions = selected_predictions[ordered_idx]
indices = indices[ordered_idx]

In [11]:
# AttrSim evaluation (same selected samples, but with token attributions as explanations)

saliency = Saliency(
    model=model,
    tokenizer=tokenizer,
    batch_size=8,
    device=device,
)

# Build attributions on the selected samples for each sample predicted class.
attr_outputs = saliency.explain(
    samples,
    targets=selected_predictions,
)

attr_system_prompt, attr_user_prompts, attr_model_predictions = attrsim.construct_prompt(
    setting=AttrSim.prompt_types.E1_attribution_with_lp,
    interesting_samples=samples,
    corresponding_predictions=selected_predictions,
    corresponding_labels=selected_labels,
    nb_learning_samples=20,
    corresponding_attribution=attr_outputs,
)

In [12]:
llm = HuggingFaceLLM(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    batch_size=2,
    device=device,
)


attr_responses = llm.batch_generate(
    attr_system_prompt,
    attr_user_prompts,
    max_new_tokens=16,
    do_sample=False,
)

attr_score = attrsim.score_from_responses(attr_responses, attr_model_predictions)

print("AttrSim score:", attr_score)
print("AttrSim responses preview:", attr_responses[:2])

AttrSim score: 0.2
AttrSim responses preview: ['The evaluation sample provided is a news article about the Nikkei stock index falling', 'of 3 minutes 13.17 seconds.\n\tLabel: \nassistant\nThe evaluation sample is a sentence from a news article about the 200']


In [15]:
attr_system_prompt

'You are a classifier. Predict the class for each evaluation sample.\n\nUse the provided learning examples and attribution explanations to infer the model behavior.\n\nOnly return the class name, no additional text.\n\nThe classes are: [World, Sports, Business, Sci/Tech]\n\nSample_0:\n\tText: Frail Pope Ends Tiring Lourdes Pilgrimage  LOURDES, France (Reuters) - Pope John Paul, a sick man  among the sick, wound up a emotional visit to this miracle  shrine Sunday and struggled with iron determination to finish a  sermon in order to encourage others suffering around him.\n\tLabel: World\n\tAttributions: {miracle: +0.005, lourdes: +0.004, lourdes: +0.003, pilgrimage: +0.003, shrine: +0.003, pope: +0.002}\nSample_1:\n\tText: Two visions of Iraq struggle to take hold Fighting in Najaf threatened to undermine a conference to choose a national assembly.\n\tLabel: World\n\tAttributions: {iraq: +0.003, struggle: +0.001, assembly: +0.001, visions: +0.001, conference: +0.001, undermine: +0.001}\n